# Fine-Tuning Small LLMs for Function Calling

This notebook fine-tunes small language models for custom tool/function calling tasks.

## Supported Models

Any model with function calling support works, including:
- **Qwen**: `Qwen/Qwen3.5-0.8B`, `Qwen/Qwen3-0.6B`, `Qwen/Qwen2.5-0.5B`
- **FunctionGemma**: `google/functiongemma-270m-it`, `google/functiongemma-2b-it`

Checkout [TRL documentation](https://github.com/huggingface/trl) for more supported models.

## Input Data

The input parameter `TRAINING_DATA_URL` accepts an URL to a file that contains a JSON object with the following fields:

1. **data**: a list of JSON objects with fields `{prompt, tool, parameters}` objects
2. **tools** - List of tool definitions in OpenAI JSON schema format

## Output

- Fine-tuned model in Hugging Face format

## Running as a Kubeflow Pipeline

This notebook can run as a Kubeflow Pipeline, but it is required to set the production PyPi repository with flag `--break-system-packages`. Hence before running Jupyter Lab with Kale or using the `kale` command the env var `KALE_PYPI_PROD_URL` should be correctly configured:
```
export KALE_PYPI_PROD_URL="https://pypi.org/simple --break-system-packages"
```


## Table of Contents

* [Step 0](#step0): Install Dependencies
* [Step 1](#step1): Load Training Data and Tool Definitions
* [Step 2](#step2): Prepare Dataset for Fine-Tuning
* [Step 3](#step3): Load Model and Tokenizer
* [Step 4](#step4): Train the Model

<a id='step0'></a>
## Step 0: Install Dependencies

Install the required libraries for fine-tuning

In [1]:
!pip install torch tensorboard
!pip install transformers datasets accelerate evaluate trl protobuf sentencepiece
!pip install requests

  Using cached protobuf-7.35.1-cp310-abi3-manylinux2014_x86_64.whl.metadata (595 bytes)
Using cached protobuf-7.35.1-cp310-abi3-manylinux2014_x86_64.whl (327 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.25.9
    Uninstalling protobuf-4.25.9:
      Successfully uninstalled protobuf-4.25.9
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
kfp-pipeline-spec 2.17.0 requires protobuf<7.0,>=6.31.1, but you have protobuf 7.35.1 which is incompatible.
google-api-core 2.34.0 requires requests<3.0.0,>=2.33.0, but you have requests 2.32.5 which is incompatible.
kfp 2.17.0 requires protobuf<7.0,>=6.31.1, but you have protobuf 7.35.1 which is incompatible.
kfp 2.17.0 requires requests==2.33.0; python_version >= "3.10", but you have requests 2.32.5 which is incompatible.
kfp-kubernetes 2.17.0 requires protobuf<7.0,>=6.33.5, but you have prot

In [2]:
import json
import os
import subprocess
import torch
import requests
import shutil
import hashlib

from collections import Counter
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import SFTConfig, SFTTrainer

### Pipeline Parameters

In [3]:
# Choose your base model
MODEL_ID = "Qwen/Qwen2.5-0.5B"
SYSTEM_MESSAGE = "You are a helpful assistant that calls tools based on user requests."
TRAINING_DATA_URL = ""

LEARNING_RATE = 3e-5
BATCH_SIZE = 1
NUM_EPOCHS = 3
MAX_LENGTH = 1024
TEST_SIZE = 0.2
SAVE_MEMORY=True

<a id='step1'></a>
## Step 1: Load Training Data and Tool Definitions

### Expected File Formats

**training_data.json:**
```json
{
  "tools": [
    {
      "type": "function",
      "function": {
        "name": "run_notebook",
        "description": "Execute a Jupyter notebook as a Kubeflow pipeline run",
        "parameters": {
          "type": "object",
          "properties": {
            "path_to_notebook": {
              "type": "string",
              "description": "Path to the notebook to execute"
            }
          },
          "required": [
            "path_to_notebook"
          ]
        }
      }
    }
  ],
  "data": [
    {
      "prompt": "Run my fraud detection notebook as a pipeline",
      "tool": "run_notebook",
      "parameters": {
        "path_to_notebook": "notebooks/fraud_detection.ipynb"
      }
    },
    {
      "prompt": "Show me all available pipelines",
      "tool": "list_pipelines",
      "parameters": {}
    }
  ]
}
```

In [4]:
# This is the data format that should be on the remote file
data = {
    "tools": [
        {
            "type": "function",
            "function": {
                "name": "run_notebook",
                "description": "Execute a Jupyter notebook as a Kubeflow pipeline run.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "path_to_notebook": {"type": "string", "description": "Path to the Jupyter notebook to execute as a pipeline"}
                    },
                    "required": ["path_to_notebook"]
                }
            }
        },
        {
            "type": "function",
            "function": {
                "name": "list_pipelines",
                "description": "List all available Kubeflow pipelines.",
                "parameters": {
                    "type": "object",
                    "properties": {},
                    "required": []
                }
            }
        },
        {
            "type": "function",
            "function": {
                "name": "list_runs",
                "description": "List all pipeline runs in Kubeflow Pipelines.",
                "parameters": {
                    "type": "object",
                    "properties": {},
                    "required": []
                }
            }
        }
    ],
    
    "data": [
        # run_notebook samples
        {"prompt": "Run my fraud detection notebook as a pipeline", "tool": "run_notebook", "parameters": {"path_to_notebook": "notebooks/fraud_detection.ipynb"}},
        {"prompt": "Execute the model training notebook", "tool": "run_notebook", "parameters": {"path_to_notebook": "notebooks/model_training.ipynb"}},
        {"prompt": "I want to run the data preprocessing notebook", "tool": "run_notebook", "parameters": {"path_to_notebook": "notebooks/data_prep.ipynb"}},
        {"prompt": "Please run the image classification notebook through the pipeline", "tool": "run_notebook", "parameters": {"path_to_notebook": "notebooks/image_classifier.ipynb"}},
        {"prompt": "Launch the customer churn prediction notebook", "tool": "run_notebook", "parameters": {"path_to_notebook": "notebooks/churn_prediction.ipynb"}},
        {"prompt": "Can you execute my sentiment analysis notebook?", "tool": "run_notebook", "parameters": {"path_to_notebook": "notebooks/sentiment_analysis.ipynb"}},
        {"prompt": "Run the feature engineering notebook as a pipeline", "tool": "run_notebook", "parameters": {"path_to_notebook": "notebooks/feature_engineering.ipynb"}},
        {"prompt": "Start a pipeline run from my recommendation system notebook", "tool": "run_notebook", "parameters": {"path_to_notebook": "notebooks/recommendation_system.ipynb"}},
        {"prompt": "I need to execute the time series forecasting notebook", "tool": "run_notebook", "parameters": {"path_to_notebook": "notebooks/time_series.ipynb"}},
        {"prompt": "Kick off the anomaly detection notebook", "tool": "run_notebook", "parameters": {"path_to_notebook": "notebooks/anomaly_detection.ipynb"}},
    
        # list_pipelines samples
        {"prompt": "Show me all available pipelines", "tool": "list_pipelines", "parameters": {}},
        {"prompt": "What pipelines do we have?", "tool": "list_pipelines", "parameters": {}},
        {"prompt": "List the pipelines", "tool": "list_pipelines", "parameters": {}},
        {"prompt": "Can you show me the existing pipelines?", "tool": "list_pipelines", "parameters": {}},
        {"prompt": "I want to see all pipelines in Kubeflow", "tool": "list_pipelines", "parameters": {}},
        {"prompt": "Give me a list of all the pipelines", "tool": "list_pipelines", "parameters": {}},
        {"prompt": "What pipelines are currently available?", "tool": "list_pipelines", "parameters": {}},
        {"prompt": "Display all pipelines", "tool": "list_pipelines", "parameters": {}},
        {"prompt": "Which pipelines exist right now?", "tool": "list_pipelines", "parameters": {}},
    
        # list_runs samples
        {"prompt": "Show me all pipeline runs", "tool": "list_runs", "parameters": {}},
        {"prompt": "What runs are currently active?", "tool": "list_runs", "parameters": {}},
        {"prompt": "List all the runs", "tool": "list_runs", "parameters": {}},
        {"prompt": "Can you show me the status of all runs?", "tool": "list_runs", "parameters": {}},
        {"prompt": "I want to see all pipeline executions", "tool": "list_runs", "parameters": {}},
        {"prompt": "Give me a list of all runs in Kubeflow", "tool": "list_runs", "parameters": {}},
        {"prompt": "What runs do we have?", "tool": "list_runs", "parameters": {}},
        {"prompt": "Display the pipeline runs", "tool": "list_runs", "parameters": {}},
    ]
}

In [5]:
device = "CPU" if not torch.cuda.is_available() else f"CUDA : {torch.cuda.get_device_name(0)}"
print(f"Environment: \nPyTorch version: {torch.__version__}. Device: {device}\n")


if TRAINING_DATA_URL.strip() and TRAINING_DATA_URL.lower().startswith("http"):
    print(f"Downloading data from {TRAINING_DATA_URL}\n")
    data = requests.get(TRAINING_DATA_URL).json()
else:
    print("Training data not provided. Using Sample Data\n")

tools = data["tools"]
training_data = data["data"]


print(f"Loaded {len(tools)} tool definitions:")
for tool in tools:
    print(f"  - {tool['function']['name']}: {tool['function']['description'][:50]}...")

required_fields = {"prompt", "tool", "parameters"}
for i, sample in enumerate(training_data):
    missing = required_fields - set(sample.keys())
    if missing:
        raise ValueError(f"Sample {i} missing required fields: {missing}")

print(f"Loaded {len(training_data)} training samples")
print(f"\nTool distribution:")
tool_counts = Counter(sample["tool"] for sample in training_data)
for tool, count in tool_counts.items():
    print(f"  {tool}: {count}")

print(f"\nFirst sample:")
print(json.dumps(training_data[0], indent=2))

Environment: 
PyTorch version: 2.11.0+cpu. Device: CPU

Training data not provided. Using Sample Data

Loaded 3 tool definitions:
  - run_notebook: Execute a Jupyter notebook as a Kubeflow pipeline ...
  - list_pipelines: List all available Kubeflow pipelines....
  - list_runs: List all pipeline runs in Kubeflow Pipelines....
Loaded 27 training samples

Tool distribution:
  run_notebook: 10
  list_pipelines: 9
  list_runs: 8

First sample:
{
  "prompt": "Run my fraud detection notebook as a pipeline",
  "tool": "run_notebook",
  "parameters": {
    "path_to_notebook": "notebooks/fraud_detection.ipynb"
  }
}


<a id='step2'></a>
## Step 2: Prepare Dataset for Fine-Tuning

Transform the JSON data into the LLM conversational format with tool calls.

In [6]:
def create_conversation(sample: dict, tools: list, system_message: str) -> dict:
    """Transform a training sample into chat format with tool calls."""
    return {
        "messages": [
            {"role": "system", "content": system_message},
            {"role": "user", "content": sample["prompt"]},
            {
                "role": "assistant",
                "tool_calls": [{
                    "type": "function",
                    "function": {"name": sample["tool"], "arguments": sample["parameters"]}
                }]
            },
        ],
        "tools": tools
    }

dataset = Dataset.from_list(training_data)

dataset = dataset.map(
    lambda sample: create_conversation(sample, tools, SYSTEM_MESSAGE),
    remove_columns=dataset.features,
    batched=False
)

dataset = dataset.train_test_split(test_size=TEST_SIZE, shuffle=True, seed=42)

print(f"Training samples: {len(dataset['train'])}")
print(f"Test samples: {len(dataset['test'])}")

Map:   0%|          | 0/27 [00:00<?, ? examples/s]

Training samples: 21
Test samples: 6


<a id='step3'></a>
## Step 3: Load Model and Tokenizer

Load the pre-trained model.
**Note**: For Function Gemma accept the Gemma terms on Hugging Face before running.

In [7]:
# Used for local notebooks execution
from huggingface_hub import notebook_login
notebook_login()

In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

In [9]:
torch_dtype = torch.float32
if torch.cuda.is_available():
    torch_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch_dtype,
    device_map="auto"
)

`torch_dtype` is deprecated! Use `dtype` instead!


<a id='step4'></a>
## Step 4: Train the Model

In [10]:
sample = dataset['train'][0]
formatted = tokenizer.apply_chat_template(
    sample["messages"],
    tools=sample["tools"],
    tokenize=False
)
print("Formatted training example:")
print(formatted[:1200] + "..." if len(formatted) > 1200 else formatted)

Formatted training example:
<|im_start|>system
You are a helpful assistant that calls tools based on user requests.

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "run_notebook", "description": "Execute a Jupyter notebook as a Kubeflow pipeline run.", "parameters": {"type": "object", "properties": {"path_to_notebook": {"type": "string", "description": "Path to the Jupyter notebook to execute as a pipeline"}}, "required": ["path_to_notebook"]}}}
{"type": "function", "function": {"name": "list_pipelines", "description": "List all available Kubeflow pipelines.", "parameters": {"type": "object", "properties": {}, "required": []}}}
{"type": "function", "function": {"name": "list_runs", "description": "List all pipeline runs in Kubeflow Pipelines.", "parameters": {"type": "object", "properties": {}, "required": []}}}
</tools>

For each fun

In [11]:
trainer = SFTTrainer(
    model=model,
    args=SFTConfig(
        output_dir="finetuned-model",
        max_length=int(MAX_LENGTH),
        packing=False,
        num_train_epochs=int(NUM_EPOCHS),
        per_device_train_batch_size=int(BATCH_SIZE),
        per_device_eval_batch_size=int(BATCH_SIZE),    
        optim="adamw_torch",
        logging_steps=1,
        eval_strategy="epoch",
        save_strategy="no",
        learning_rate=float(LEARNING_RATE),
        fp16=True if model.dtype == torch.float16 else False,
        bf16=True if model.dtype == torch.bfloat16 else False,
        lr_scheduler_type="constant",
        # Use this if you want to save a report of the trainning process (requires the tensorboard dep)
        #report_to="tensorboard",
        load_best_model_at_end=False,
        metric_for_best_model="eval_loss",
        # set these for False for more speed and resource usage
        gradient_checkpointing=SAVE_MEMORY,
    ),
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    processing_class=tokenizer,
)

if hasattr(model.config, "num_local_experts"):
  # Granite uses num_local_experts; SFTTrainer looks for num_experts
  model.config.num_experts = model.config.num_local_experts

print("Training configuration:")
print(f"  - Epochs: {trainer.args.num_train_epochs}")
print(f"  - Batch size: {trainer.args.per_device_train_batch_size}")
print(f"  - Learning rate: {trainer.args.learning_rate}")
print(f"  - Max length: {trainer.args.max_length}")

print("Trainer initialized! Starting training...")

train_result = trainer.train()

print(f"\nTraining completed!")
print(train_result)

eval_results = trainer.evaluate()
print(eval_results)
print(f"Evaluation loss: {eval_results['eval_loss']:.4f}")
# the model will be returned as output artifact
serialized_model = trainer.model
serialized_model

Tokenizing train dataset:   0%|          | 0/21 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/21 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/21 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/21 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

...

The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Training configuration:
  - Epochs: 2
  - Batch size: 1
  - Learning rate: 3e-05
  - Max length: 1024
Trainer initialized! Starting training...


/home/wsiqueir/projects/kale_examples/.examples/lib64/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.239600,0.230107,0.377464,6732.000000,0.960447


KeyboardInterrupt: 